# Data Exploration & Validation

Covers all data sources used in the composite GPR index:
- **GDELT** lexical panel + URL corpus (FinBERT sentiment)
- **ICEWS** yearly event files (1995–2022)
- **Phoenix SWB** (BBC, 1990–2019) and **Phoenix NYT** (1990–2018, CHN-USA only)
- **UCDP** dyadic conflict (evaluated, excluded)

Every expensive step (model downloads, full-file scans, aggregated CSVs) is
**skipped if the output already exists** (you may re-run freely without repeating work.)

**🕛 Total Runtime:** ~02 minutes


In [56]:
import os, glob, json, zipfile
from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np

BASE = Path.cwd().parent
GDELT_DIR   = BASE / "data" / "03_nlp" / "gdelt"
PHOENIX_DIR = BASE / "data" / "03_nlp" / "phoenix"
ICEWS_DIR   = BASE / "data" / "03_nlp" / "icews"
URLS_DIR    = GDELT_DIR / "data_nlp_urls"
UCDP_PATH   = BASE / "data" / "03_nlp" / "Dyadic_v25_1.csv"

# Looping through paths and print status using only the folder/file name
for p in [GDELT_DIR, PHOENIX_DIR, ICEWS_DIR, URLS_DIR, UCDP_PATH]:
    status = "OK  " if p.exists() else "MISS"
    print(f"{status} {p.name}")

# Canonical dyad definitions used throughout
DYADS_FULL_ICEWS = [         
    ("China", "United States"),  ("China", "Japan"),
    ("China", "Australia"),      ("China", "France"),
    ("China", "Germany"),        ("China", "United Kingdom"),
    ("China", "Russia"),         ("China", "India"),   
    ("China", "Indonesia"),      ("China", "Pakistan"),
    ("China", "Vietnam"),
]
# ICEWS full name → ISO label used everywhere else
DYAD_LABELS = {               
    "China-United States":  "CHN-USA",  "China-Japan":       "CHN-JPN",
    "China-Australia":      "CHN-AUS",  "China-France":      "CHN-FRA",
    "China-Germany":        "CHN-DEU",  "China-United Kingdom":"CHN-GBR",
    "China-Russia":         "CHN-RUS",  "China-India":       "CHN-IND",
    "China-Indonesia":      "CHN-IDN",  "China-Pakistan":    "CHN-PAK",
    "China-Vietnam":        "CHN-VNM",
}
DYADS_ISO = list(DYAD_LABELS.values())
print(f"\n{len(DYADS_ISO)} dyads configured:", DYADS_ISO)

OK   gdelt
OK   phoenix
OK   icews
OK   data_nlp_urls
OK   Dyadic_v25_1.csv

11 dyads configured: ['CHN-USA', 'CHN-JPN', 'CHN-AUS', 'CHN-FRA', 'CHN-DEU', 'CHN-GBR', 'CHN-RUS', 'CHN-IND', 'CHN-IDN', 'CHN-PAK', 'CHN-VNM']


---
## Section 1: GDELT Lexical Panel

Pre-built monthly panel of CAMEO conflict/cooperation event counts.
We verify shape, date range, dyad coverage, and key column distributions.

In [57]:
gdelt_csv = GDELT_DIR / "gdelt_birectional_lexical_panel.csv"
df_gdelt  = pd.read_csv(gdelt_csv)
df_gdelt['date'] = pd.to_datetime(df_gdelt['date'])

print("=== GDELT PANEL ===")
print(f"Shape            : {df_gdelt.shape}   (expected 4246 = 386 months × 11 dyads)")
print(f"Date range       : {df_gdelt['date'].min().date()} → {df_gdelt['date'].max().date()}")
print(f"Dyads ({df_gdelt['dyad'].nunique()})       : {sorted(df_gdelt['dyad'].unique())}")
print(f"\nNull counts:\n{df_gdelt.isnull().sum()}")
print(f"\nDescriptive stats (numeric cols):")
print(df_gdelt.describe().round(3))

=== GDELT PANEL ===
Shape            : (4246, 8)   (expected 4246 = 386 months × 11 dyads)
Date range       : 1990-01-01 → 2022-02-01
Dyads (11)       : ['CHN-AUS', 'CHN-DEU', 'CHN-FRA', 'CHN-GBR', 'CHN-IDN', 'CHN-IND', 'CHN-JPN', 'CHN-PAK', 'CHN-RUS', 'CHN-USA', 'CHN-VNM']

Null counts:
date                        0
dyad                        0
total_volume                0
url_conflict_hits           0
url_cooperation_hits        0
cameo_conflict_hits         0
cameo_cooperation_hits      0
goldstein_mean            137
dtype: int64

Descriptive stats (numeric cols):
                                date  total_volume  url_conflict_hits  \
count                           4246      4246.000           4246.000   
mean   2006-01-15 09:38:14.300518016      1021.310             53.617   
min              1990-01-01 00:00:00         0.000              0.000   
25%              1998-01-01 00:00:00        54.000              0.000   
50%              2006-01-16 12:00:00       272.000        

In [58]:
# GDELT score distribution: how often is net sentiment negative vs positive?
df_gdelt['net_ratio'] = ((df_gdelt['cameo_conflict_hits'] - df_gdelt['cameo_cooperation_hits'])
                         / df_gdelt['total_volume'].replace(0, np.nan))

import matplotlib
matplotlib.use('Agg')  # non-interactive backend for script runs
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df_gdelt['net_ratio'].hist(ax=axes[0], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('GDELT net conflict ratio (all dyads)'); axes[0].set_xlabel('(conflict-coop)/total')

df_gdelt.groupby('dyad')['total_volume'].mean().sort_values().plot.barh(ax=axes[1], color='teal')
axes[1].set_title('Mean monthly event volume by dyad'); axes[1].set_xlabel('avg events/month')

df_gdelt.groupby(df_gdelt['date'].dt.year)['total_volume'].sum().plot(ax=axes[2], color='navy')
axes[2].set_title('Total GDELT events per year (all dyads)'); axes[2].set_xlabel('year')

plt.tight_layout()
plt.savefig(GDELT_DIR / 'explore_gdelt_panel.png', dpi=130, bbox_inches='tight')
plt.show()
print("Saved: explore_gdelt_panel.png")

Saved: explore_gdelt_panel.png


C:\Users\HP\AppData\Local\Temp\ipykernel_10328\2116496006.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Section 2: GDELT URL Corpus (FinBERT Sentiment Input)

One JSONL file per dyad under `data_nlp_urls/`.
We count records, inspect date coverage, and check text quality.

In [59]:
jsonl_files = sorted(URLS_DIR.glob("corpus_*.jsonl"))
print(f"Found {len(jsonl_files)} JSONL files:\n")

url_summary = []
for jf in jsonl_files:
    dyad = jf.stem.replace("corpus_","")
    size_mb = jf.stat().st_size / 1e6
    # Count lines fast without loading everything
    n_lines = sum(1 for _ in open(jf, encoding='utf-8', errors='ignore'))
    url_summary.append({'dyad': dyad, 'size_mb': round(size_mb,1), 'n_records': n_lines})
    print(f"  {dyad:12s} {size_mb:7.1f} MB   {n_lines:>9,} records")

df_url_summary = pd.DataFrame(url_summary)
print(f"\nTotal records: {df_url_summary['n_records'].sum():,}")

Found 11 JSONL files:

  CHN-AUS         53.6 MB     240,401 records
  CHN-DEU         22.6 MB     109,809 records
  CHN-FRA         27.5 MB     133,420 records
  CHN-GBR         57.1 MB     267,349 records
  CHN-IDN         12.9 MB      59,977 records
  CHN-IND         18.2 MB      79,873 records
  CHN-JPN         65.2 MB     319,643 records
  CHN-PAK         47.8 MB     224,420 records
  CHN-RUS         75.8 MB     367,985 records
  CHN-USA        267.3 MB   1,245,202 records
  CHN-VNM         28.5 MB     136,386 records

Total records: 3,184,465


In [60]:
# Inspect one record to understand field structure
sample_jf = URLS_DIR / "corpus_CHN-USA.jsonl"
print("=== Sample records from CHN-USA corpus ===\n")
with open(sample_jf, encoding='utf-8', errors='ignore') as f:
    for i, line in enumerate(f):
        if i >= 5: break
        obj = json.loads(line)
        print(f"Record {i}: keys={list(obj.keys())}")
        print(f"  date     : {obj.get('date','')}")
        print(f"  day_key  : {obj.get('day_key','')}")
        print(f"  text[:80]: {str(obj.get('cleaned_text',''))[:80]}")
        print()

=== Sample records from CHN-USA corpus ===

Record 0: keys=['date', 'day_key', 'raw_url', 'cleaned_text']
  date     : 2013-04-01
  day_key  : 20130401
  text[:80]: 

Record 1: keys=['date', 'day_key', 'raw_url', 'cleaned_text']
  date     : 2013-04-01
  day_key  : 20130401
  text[:80]: 

Record 2: keys=['date', 'day_key', 'raw_url', 'cleaned_text']
  date     : 2013-04-01
  day_key  : 20130401
  text[:80]: 

Record 3: keys=['date', 'day_key', 'raw_url', 'cleaned_text']
  date     : 2013-04-01
  day_key  : 20130401
  text[:80]: story rss

Record 4: keys=['date', 'day_key', 'raw_url', 'cleaned_text']
  date     : 2013-04-01
  day_key  : 20130401
  text[:80]: story rss



In [61]:
# Date coverage per dyad — how many months have data?
print("=== Monthly coverage per dyad (first and last month) ===\n")
for jf in jsonl_files:
    dyad  = jf.stem.replace("corpus_","")
    dates = set()
    with open(jf, encoding='utf-8', errors='ignore') as f:
        for line in f:
            try:
                obj  = json.loads(line)
                d    = obj.get('date','') or (str(obj.get('day_key',''))[:6].replace('','') )
                if d: dates.add(d[:7])   # YYYY-MM
            except: pass
    dates = sorted(dates)
    print(f"  {dyad:12s}  {len(dates):3d} months   {dates[0] if dates else 'n/a'} → {dates[-1] if dates else 'n/a'}")

=== Monthly coverage per dyad (first and last month) ===

  CHN-AUS       107 months   2013-04 → 2022-02
  CHN-DEU       107 months   2013-04 → 2022-02
  CHN-FRA       107 months   2013-04 → 2022-02
  CHN-GBR       107 months   2013-04 → 2022-02
  CHN-IDN       107 months   2013-04 → 2022-02
  CHN-IND       107 months   2013-04 → 2022-02
  CHN-JPN       107 months   2013-04 → 2022-02
  CHN-PAK       107 months   2013-04 → 2022-02
  CHN-RUS       107 months   2013-04 → 2022-02
  CHN-USA       107 months   2013-04 → 2022-02
  CHN-VNM       107 months   2013-04 → 2022-02


---
## Section 3: ICEWS Raw File Audit

### 3a. File inventory and column names

We read the first 5 rows of three representative years to confirm:
- Column names (especially `Source Country`, `Target Country`, `Intensity`)
- How Russia is spelled ("Russia" vs "Russia (USSR)")
- Whether all 11 dyads are present

**Why this matters:** the merge bug that produced CHN-RUS = 0% was an exact-match
on `"Russia"` which missed `"Russia (USSR)"` in early files (1995–2004).

In [62]:
def read_icews_file(filepath, nrows=None):
    """Read ICEWS file (.tab, .tsv, or .zip containing either)."""
    fp = str(filepath)
    if fp.endswith('.zip'):
        with zipfile.ZipFile(filepath) as zf:
            with zf.open(zf.namelist()[0]) as f:
                return pd.read_csv(f, sep='\t', nrows=nrows, dtype=str, low_memory=False)
    return pd.read_csv(filepath, sep='\t', nrows=nrows, dtype=str, low_memory=False)

icews_files = sorted(
    glob.glob(str(ICEWS_DIR / "events.*.tab*")) +
    glob.glob(str(ICEWS_DIR / "events.*.tsv*"))
)
print(f"Found {len(icews_files)} ICEWS yearly files\n")

# Probe 3 representative years
probe_names = {"1995", "2004", "2010", "2021"}
for fp in icews_files:
    yr = Path(fp).name.split('.')[1]
    if yr not in probe_names: continue
    print(f"\n--- {Path(fp).name} ---")
    df = read_icews_file(fp, nrows=5000)
    print(f"  Columns : {df.columns.tolist()}")
    # Country name variants for our key countries
    for country in ["China","Russia","United States","India","Vietnam"]:
        exact   = (df['Source Country'].eq(country) | df['Target Country'].eq(country)).sum()
        partial = (df['Source Country'].str.contains(country,na=False) |
                   df['Target Country'].str.contains(country,na=False)).sum()
        if partial > 0:
            uniq = list(pd.concat([
                df.loc[df['Source Country'].str.contains(country,na=False),'Source Country'],
                df.loc[df['Target Country'].str.contains(country,na=False),'Target Country']
            ]).unique()[:4])
            print(f"  {country:<18}: exact={exact:4d}  partial={partial:4d}  vals={uniq}")

Found 27 ICEWS yearly files


--- events.1995.20150313082510.tab.zip ---
  Columns : ['Event ID', 'Event Date', 'Source Name', 'Source Sectors', 'Source Country', 'Event Text', 'CAMEO Code', 'Intensity', 'Target Name', 'Target Sectors', 'Target Country', 'Story ID', 'Sentence Number', 'Publisher', 'City', 'District', 'Province', 'Country', 'Latitude', 'Longitude']
  China             : exact=  86  partial=  86  vals=['China']
  Russia            : exact=   0  partial= 659  vals=['Russian Federation']
  United States     : exact=1169  partial=1169  vals=['United States']
  India             : exact= 117  partial= 117  vals=['India']
  Vietnam           : exact=  27  partial=  27  vals=['Vietnam']

--- events.2004.20150313083407.tab.zip ---
  Columns : ['Event ID', 'Event Date', 'Source Name', 'Source Sectors', 'Source Country', 'Event Text', 'CAMEO Code', 'Intensity', 'Target Name', 'Target Sectors', 'Target Country', 'Story ID', 'Sentence Number', 'Publisher', 'City', 'District', 'Prov

### 3b. Full dyad-year event count audit

Read every ICEWS file completely and count events per dyad per year.
Uses `str.contains` (not `==`) so "Russia (USSR)" is captured.

**Skip logic:** if `icews_audit_counts.csv` already exists, load it instead of reprocessing.
The full scan takes ~10 minutes.

In [63]:
ICEWS_AUDIT_PATH = ICEWS_DIR / "icews_audit_counts.csv"

if ICEWS_AUDIT_PATH.exists():
    print(f"Loading cached audit: {ICEWS_AUDIT_PATH}")
    df_audit = pd.read_csv(ICEWS_AUDIT_PATH, index_col=0)
else:
    from tqdm import tqdm
    audit = defaultdict(lambda: defaultdict(int))   # label -> year -> count

    for fp in tqdm(icews_files, desc="ICEWS audit"):
        yr_str = Path(fp).name.split('.')[1]
        if not yr_str.isdigit(): continue
        year = int(yr_str)
        if year < 1995 or year > 2022: continue
        try:
            df = read_icews_file(fp)
        except Exception as e:
            print(f"  Error {fp}: {e}"); continue

        for s_pat, t_pat in DYADS_FULL_ICEWS:
            label = DYAD_LABELS[f"{s_pat}-{t_pat}"]
            mask = (
                (df['Source Country'].str.contains(s_pat, na=False) &
                 df['Target Country'].str.contains(t_pat, na=False)) |
                (df['Source Country'].str.contains(t_pat, na=False) &
                 df['Target Country'].str.contains(s_pat, na=False))
            )
            audit[label][year] += mask.sum()

    df_audit = pd.DataFrame(audit).T.fillna(0).astype(int)
    df_audit = df_audit.reindex(sorted(df_audit.columns), axis=1)
    df_audit.to_csv(ICEWS_AUDIT_PATH)
    print(f"Saved audit cache: {ICEWS_AUDIT_PATH}")

print("\n=== ICEWS event counts per dyad × year ===")
print(df_audit.to_string())

Loading cached audit: c:\Users\HP\Desktop\macro-geopolitics\data\03_nlp\icews\icews_audit_counts.csv

=== ICEWS event counts per dyad × year ===
         1995  1996  1997  1998  1999  2000  2001  2002  2003  2004  2005   2006  2007  2008  2009  2010  2011  2012  2013  2014  2015  2016  2018   2019  2020  2021  2022
CHN-USA  1411  2024  3779  5115  5968  3444  7297  6057  6869  6045  9507  10987  7496  6419  8337  6666  6000  4707  4031  5308  5911  5480  7615  10160  6718  9645  8875
CHN-JPN   305   625  1049  2065  1792  2809  2719  3827  4171  4643  8005   8757  8961  6102  3931  4524  2423  2265  2385  3352  3216  2034  2561   1709  1251  1145  1436
CHN-AUS    80   365   373   199   975   207   245   865  1087   409  1209   1128  1481  1567  1104   559   440   284   559  1102   433   689   647    621  1128   698  1707
CHN-FRA    36   321   321   529   512   543   518   320   799  1415   729   1250  1108  1551   932   975   614   336   574   678   820   240   404    469   317   476  

In [64]:
# Visualise coverage heatmap
fig, ax = plt.subplots(figsize=(16, 5))
import matplotlib.colors as mcolors
cmap = plt.cm.YlOrRd
im = ax.imshow(df_audit.values, aspect='auto', cmap=cmap,
               norm=mcolors.LogNorm(vmin=1, vmax=df_audit.values.max()))
ax.set_yticks(range(len(df_audit.index)))
ax.set_yticklabels(df_audit.index, fontsize=9)
ax.set_xticks(range(len(df_audit.columns)))
ax.set_xticklabels(df_audit.columns, rotation=45, ha='right', fontsize=8)
plt.colorbar(im, ax=ax, label='Event count (log scale)')
ax.set_title('ICEWS event counts per dyad × year (log scale)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(ICEWS_DIR / 'explore_icews_coverage.png', dpi=130, bbox_inches='tight')
plt.show()
print("\nZero-count cells (potential merge gaps):")
zeros = [(dyad, yr) for dyad in df_audit.index for yr in df_audit.columns if df_audit.loc[dyad,yr]==0]
for d,y in zeros: print(f"  {d} {y}")


Zero-count cells (potential merge gaps):


C:\Users\HP\AppData\Local\Temp\ipykernel_10328\3546649254.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 3c. ICEWS aggregation to monthly panel

Produces `icews_monthly_panel_1995_2022.csv` in the `icews/` folder.

**Skipped if the file already exists** — delete it to force reprocessing.

In [65]:
ICEWS_PANEL_PATH = ICEWS_DIR / "icews_monthly_panel_1995_2022.csv"

if ICEWS_PANEL_PATH.exists():
    print(f"Loading existing ICEWS panel: {ICEWS_PANEL_PATH}")
    df_icews = pd.read_csv(ICEWS_PANEL_PATH)
    df_icews['date'] = pd.to_datetime(df_icews['date'])
    print(f"Shape: {df_icews.shape}")
    print(f"Dyads: {df_icews['dyad'].unique().tolist()}")
    print(f"Date range: {df_icews['date'].min().date()} → {df_icews['date'].max().date()}")
    print(f"\nMonths per dyad:\n{df_icews['dyad'].value_counts()}")
else:
    from tqdm import tqdm
    all_months = []
    for fp in tqdm(icews_files, desc="ICEWS aggregation"):
        yr_str = Path(fp).name.split('.')[1]
        if not yr_str.isdigit(): continue
        year = int(yr_str)
        if year < 1995 or year > 2022: continue
        try:
            df = read_icews_file(fp)
        except Exception as e:
            print(f"  Error {fp}: {e}"); continue

        df['event_date'] = pd.to_datetime(df['Event Date'], errors='coerce')
        df['year']  = df['event_date'].dt.year
        df['month'] = df['event_date'].dt.month
        df = df.dropna(subset=['year','month'])
        df['Intensity'] = pd.to_numeric(df['Intensity'], errors='coerce')

        for s_pat, t_pat in DYADS_FULL_ICEWS:
            label = DYAD_LABELS[f"{s_pat}-{t_pat}"]
            mask = (
                (df['Source Country'].str.contains(s_pat, na=False) &
                 df['Target Country'].str.contains(t_pat, na=False)) |
                (df['Source Country'].str.contains(t_pat, na=False) &
                 df['Target Country'].str.contains(s_pat, na=False))
            )
            dyad_df = df[mask].copy()
            if dyad_df.empty: continue
            monthly = dyad_df.groupby(['year','month']).agg(
                icews_event_count   =('Event ID', 'count'),
                icews_goldstein_mean=('Intensity', 'mean'),
                icews_goldstein_std =('Intensity', 'std'),
            ).reset_index()
            monthly['date'] = pd.to_datetime(monthly[['year','month']].assign(day=1))
            monthly['dyad'] = label
            all_months.append(monthly)

    df_icews = pd.concat(all_months, ignore_index=True) if all_months else pd.DataFrame()
    if not df_icews.empty:
        df_icews = df_icews.drop_duplicates(subset=['date','dyad'])
        df_icews.to_csv(ICEWS_PANEL_PATH, index=False)
        print(f"Saved → {ICEWS_PANEL_PATH}  ({df_icews.shape})")
        print(f"\nMonths per dyad:\n{df_icews['dyad'].value_counts()}")

Loading existing ICEWS panel: c:\Users\HP\Desktop\macro-geopolitics\data\03_nlp\icews\icews_monthly_panel_1995_2022.csv
Shape: (3488, 7)
Dyads: ['CHN-USA', 'CHN-JPN', 'CHN-AUS', 'CHN-FRA', 'CHN-DEU', 'CHN-GBR', 'CHN-RUS', 'CHN-IND', 'CHN-IDN', 'CHN-PAK', 'CHN-VNM']
Date range: 1995-01-01 → 2022-12-01

Months per dyad:
dyad
CHN-USA    324
CHN-JPN    322
CHN-FRA    322
CHN-RUS    322
CHN-GBR    320
CHN-VNM    320
CHN-AUS    317
CHN-PAK    314
CHN-IND    314
CHN-DEU    309
CHN-IDN    304
Name: count, dtype: int64


In [66]:
# Goldstein distribution by dyad
if 'df_icews' in dir() and not df_icews.empty:
    fig, axes = plt.subplots(3, 4, figsize=(16, 10), sharey=False)
    axes = axes.flatten()
    for i, dyad in enumerate(sorted(df_icews['dyad'].unique())):
        sub = df_icews[df_icews['dyad']==dyad]
        axes[i].hist(sub['icews_goldstein_mean'].dropna(), bins=30,
                     color='steelblue', edgecolor='white')
        axes[i].axvline(0, color='red', lw=0.8, ls='--')
        axes[i].set_title(dyad, fontsize=9)
        n = sub['icews_event_count'].sum()
        axes[i].set_xlabel(f"Goldstein mean  (n={int(n):,} events)", fontsize=7)
    for j in range(i+1, len(axes)): axes[j].set_visible(False)
    fig.suptitle('ICEWS monthly Goldstein score distribution per dyad', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(ICEWS_DIR / 'explore_icews_goldstein.png', dpi=130, bbox_inches='tight')
    plt.show()

C:\Users\HP\AppData\Local\Temp\ipykernel_10328\3894398779.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Section 4: Phoenix SWB (BBC Summary of World Broadcasts)

### 4a. Column structure and actor code format

Phoenix uses CAMEO actor codes in `source`/`target` columns — e.g. `"CHNGOV"`,
`"RUSGOV"`, `"CHN"`, `"RUS"`.  `str.contains(ISO3)` correctly matches all variants.
We first inspect the actual values before running the full scan.

In [67]:
swb_file = PHOENIX_DIR / "PhoenixBLN-SWB_1979-2019.csv"
df_swb_sample = pd.read_csv(swb_file, nrows=100_000, low_memory=False,
                             usecols=['story_date','source','target','goldstein','eid'])

print(f"Columns : {df_swb_sample.columns.tolist()}")
print(f"Date range in first 100k rows: {pd.to_datetime(df_swb_sample['story_date'],errors='coerce').min().date()} → {pd.to_datetime(df_swb_sample['story_date'],errors='coerce').max().date()}")
print(f"\nSample source values (first 30 unique):")
print(sorted(df_swb_sample['source'].dropna().unique())[:30])
print(f"\nSample target values (first 30 unique):")
print(sorted(df_swb_sample['target'].dropna().unique())[:30])

Columns : ['eid', 'story_date', 'source', 'target', 'goldstein']
Date range in first 100k rows: 1979-11-30 → 2018-12-19

Sample source values (first 30 unique):
['---', '---AGR', '---AGRLAB', '---AGROPP', '---BUD', '---BUDPTY', '---BUDUAF', '---BUS', '---BUSCOP', '---BUSCVL', '---BUSEDU', '---BUSGOV', '---BUSGOVDEV', '---BUSGOVMED', '---BUSJEW', '---BUSLAB', '---BUSLEG', '---BUSMED', '---BUSMIL', '---BUSMILCOPHLHGOV', '---BUSOPP', '---BUSUAF', '---CHR', '---CHRAGR', '---CHRCOP', '---CHRCTH', '---CHRCTHJUD', '---CHRCVL', '---CHREDU', '---CHRGOV']

Sample target values (first 30 unique):
['---', '---AGR', '---AGRCVL', '---AGREDU', '---AGRGOV', '---AGRJUD', '---AGRLAB', '---AGRMIL', '---BUD', '---BUDMIL', '---BUDREL', '---BUS', '---BUSAGR', '---BUSCOP', '---BUSCRM', '---BUSCVL', '---BUSEDU', '---BUSGOV', '---BUSGOVEDU', '---BUSLAB', '---BUSLEG', '---BUSMED', '---BUSMIL', '---BUSMILCVL', '---BUSNGO', '---BUSPTY', '---CHR', '---CHRCOP', '---CHRCTH', '---CHRCTHCVL']


In [68]:
# Actor code audit: what forms do CHN and RUS take?
for iso in ['CHN','RUS','USA','JPN','IND']:
    src_vals = df_swb_sample.loc[df_swb_sample['source'].str.contains(iso,na=False),'source'].unique()[:8]
    tgt_vals = df_swb_sample.loc[df_swb_sample['target'].str.contains(iso,na=False),'target'].unique()[:8]
    n_src = df_swb_sample['source'].str.contains(iso,na=False).sum()
    n_tgt = df_swb_sample['target'].str.contains(iso,na=False).sum()
    print(f"{iso}  src_hits={n_src:5d}  sample_codes={list(src_vals)}")
    print(f"     tgt_hits={n_tgt:5d}  sample_codes={list(tgt_vals)}")
    print()

CHN  src_hits= 4062  sample_codes=['CHN', 'CHNBUS', 'CHNGOV', 'CHNLEG', 'CHNELI', 'CHNGOVLEG', 'CHNELILEG', 'CHNPPLGOV']
     tgt_hits= 3399  sample_codes=['CHN', 'CHNGOV', 'CHNMUSRAD', 'CHNLEG', 'CHNMILGOV', 'CHNBUS', 'CHNELI', 'CHNCOPGOV']

RUS  src_hits= 5677  sample_codes=['RUSELIGOV', 'RUS', 'RUSGOV', 'RUSMEDGOV', 'RUSCOP', 'RUSPTYGOV', 'RUSMED', 'RUSMIL']
     tgt_hits= 4866  sample_codes=['RUSGOV', 'RUSMED', 'RUS', 'RUSLEG', 'RUSCVL', 'RUSCOPMIL', 'RUSGOVCOP', 'RUSCOPGOV']

USA  src_hits= 5006  sample_codes=['USA', 'USAGOV', 'USAMILLEG', 'USAGOVLEG', 'USAGOVMIL', 'USAMIL', 'USAMILMED', 'USAPTY']
     tgt_hits= 5022  sample_codes=['USA', 'USAMILMED', 'USAGOV', 'USALEG', 'USAMILGOV', 'USAMIL', 'USAGOVLEG', 'USAMILLEG']

JPN  src_hits= 1336  sample_codes=['JPNMED', 'JPN', 'JPNGOV', 'JPNMIL', 'JPNBUS', 'JPNGOVPTY', 'JPNJUD', 'JPNPTYMED']
     tgt_hits= 1238  sample_codes=['JPN', 'JPNGOV', 'JPNMIL', 'JPNMED', 'JPNPTYMED', 'MNCJPN', 'JPNJUD', 'JPNELIGOV']

IND  src_hits=  864  sample_

### 4b. SWB dyad-year event count audit

**Skip logic:** loads cached `swb_audit_counts.csv` if present.

In [69]:
SWB_AUDIT_PATH = PHOENIX_DIR / "swb_audit_counts.csv"

if SWB_AUDIT_PATH.exists():
    print(f"Loading cached audit: {SWB_AUDIT_PATH}")
    df_swb_audit = pd.read_csv(SWB_AUDIT_PATH, index_col=0)
else:
    from tqdm import tqdm
    swb_counts = defaultdict(lambda: defaultdict(int))
    for chunk in tqdm(
        pd.read_csv(swb_file, chunksize=200_000, low_memory=False,
                    usecols=['story_date','source','target']),
        desc="SWB audit"
    ):
        chunk['year'] = pd.to_datetime(chunk['story_date'],errors='coerce').dt.year
        chunk = chunk[(chunk['year']>=1990) & (chunk['year']<=2019)]
        if chunk.empty: continue
        for iso1,iso2 in [(d[:3],d[4:]) for d in [dyn.replace('CHN-','CHN#').replace('-','X',1).replace('#','-') for dyn in DYADS_ISO]]:
            # simpler: just unpack from DYADS_ISO directly
            pass

    # Redo cleanly
    swb_counts = defaultdict(lambda: defaultdict(int))
    for chunk in tqdm(
        pd.read_csv(swb_file, chunksize=200_000, low_memory=False,
                    usecols=['story_date','source','target']),
        desc="SWB audit"
    ):
        chunk['year'] = pd.to_datetime(chunk['story_date'],errors='coerce').dt.year
        chunk = chunk[(chunk['year']>=1990) & (chunk['year']<=2019)]
        if chunk.empty: continue
        for dyad in DYADS_ISO:
            iso1, iso2 = dyad[:3], dyad[4:]
            mask = (
                (chunk['source'].str.contains(iso1,na=False) & chunk['target'].str.contains(iso2,na=False)) |
                (chunk['source'].str.contains(iso2,na=False) & chunk['target'].str.contains(iso1,na=False))
            )
            for yr, cnt in chunk.loc[mask,'year'].value_counts().items():
                swb_counts[dyad][yr] += cnt

    df_swb_audit = pd.DataFrame(swb_counts).T.fillna(0).astype(int)
    df_swb_audit = df_swb_audit.reindex(sorted(df_swb_audit.columns),axis=1)
    df_swb_audit.to_csv(SWB_AUDIT_PATH)
    print(f"Saved: {SWB_AUDIT_PATH}")

print("\n=== SWB event counts per dyad × year ===")
print(df_swb_audit.to_string())

Loading cached audit: c:\Users\HP\Desktop\macro-geopolitics\data\03_nlp\phoenix\swb_audit_counts.csv

=== SWB event counts per dyad × year ===
         1990  1991  1992  1993  1994  1995  1996  1997  1998  1999  2000  2001  2002  2003  2004  2005  2006  2007  2008  2009  2010  2011  2012  2013  2014  2015  2016  2017  2018  2019
CHN-USA    95   145   196   220   331   420   452   554  1252  1262   943  1372  1374   761   746  1002   906   740   570   998   918   910   633   443   397   431   429   348   559    99
CHN-JPN   127   132   174    88   112   198   197   322   541   385   540   653   832   473   675   777   742   807   535   368   552   361   515   326   411   316   230   134   254    15
CHN-AUS    12    13    36    16    30    42    36    57    61    89    54    26   115    94    33    71    96    97    91    91    56    40    15    44    62    19    26    18    23     4
CHN-FRA    32    41    24    10    36    34    57    50   129   130    69   122    80    91   184   102  

In [70]:
# Heatmap
fig, ax = plt.subplots(figsize=(16,4))
im = ax.imshow(df_swb_audit.values, aspect='auto', cmap='Blues')
ax.set_yticks(range(len(df_swb_audit.index))); ax.set_yticklabels(df_swb_audit.index, fontsize=9)
ax.set_xticks(range(len(df_swb_audit.columns)))
ax.set_xticklabels(df_swb_audit.columns, rotation=45, ha='right', fontsize=8)
plt.colorbar(im, ax=ax, label='Event count')
ax.set_title('Phoenix SWB coverage per dyad × year', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(PHOENIX_DIR / 'explore_swb_coverage.png', dpi=130, bbox_inches='tight')
plt.show()

print("\nDyads with ZERO SWB events across all years:")
for dyad in df_swb_audit.index:
    if df_swb_audit.loc[dyad].sum() == 0:
        print(f"  {dyad}  — genuinely absent from SWB")


Dyads with ZERO SWB events across all years:


C:\Users\HP\AppData\Local\Temp\ipykernel_10328\3074156461.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4c. SWB monthly panel aggregation

**Skipped if `phoenix_swb_monthly_1990_2019.csv` already exists.**

In [71]:
SWB_PANEL_PATH = PHOENIX_DIR / "phoenix_swb_monthly_1990_2019.csv"

if SWB_PANEL_PATH.exists():
    print(f"Loading existing SWB panel: {SWB_PANEL_PATH}")
    df_swb = pd.read_csv(SWB_PANEL_PATH)
    df_swb['date'] = pd.to_datetime(df_swb['date'])
    print(f"Shape: {df_swb.shape}")
    print(f"\nMonths per dyad:\n{df_swb['dyad'].value_counts()}")
    print(f"\nGoldstein stats per dyad:")
    print(df_swb.groupby('dyad')['swb_goldstein_mean'].describe().round(3))
else:
    from tqdm import tqdm
    all_swb = []
    for chunk in tqdm(
        pd.read_csv(swb_file, chunksize=200_000, low_memory=False,
                    usecols=['story_date','source','target','eid','goldstein']),
        desc="SWB aggregation"
    ):
        chunk['date']  = pd.to_datetime(chunk['story_date'], errors='coerce')
        chunk['year']  = chunk['date'].dt.year
        chunk['month'] = chunk['date'].dt.month
        chunk = chunk[(chunk['year']>=1990) & (chunk['year']<=2019)]
        if chunk.empty: continue
        for dyad in DYADS_ISO:
            iso1, iso2 = dyad[:3], dyad[4:]
            mask = (
                (chunk['source'].str.contains(iso1,na=False) & chunk['target'].str.contains(iso2,na=False)) |
                (chunk['source'].str.contains(iso2,na=False) & chunk['target'].str.contains(iso1,na=False))
            )
            dc = chunk[mask].copy()
            if dc.empty: continue
            monthly = dc.groupby(['year','month']).agg(
                swb_event_count    =('eid',       'count'),
                swb_goldstein_mean =('goldstein', 'mean'),
                swb_goldstein_std  =('goldstein', 'std'),
            ).reset_index()
            monthly['date'] = pd.to_datetime(monthly[['year','month']].assign(day=1))
            monthly['dyad'] = dyad
            all_swb.append(monthly)

    df_swb = pd.concat(all_swb, ignore_index=True) if all_swb else pd.DataFrame()
    if not df_swb.empty:
        df_swb = df_swb.drop_duplicates(subset=['date','dyad'])
        df_swb.to_csv(SWB_PANEL_PATH, index=False)
        print(f"Saved → {SWB_PANEL_PATH}  ({df_swb.shape})")

Loading existing SWB panel: c:\Users\HP\Desktop\macro-geopolitics\data\03_nlp\phoenix\phoenix_swb_monthly_1990_2019.csv
Shape: (3433, 7)

Months per dyad:
dyad
CHN-USA    351
CHN-JPN    351
CHN-RUS    341
CHN-PAK    325
CHN-GBR    323
CHN-VNM    315
CHN-IND    307
CHN-FRA    294
CHN-AUS    279
CHN-DEU    279
CHN-IDN    268
Name: count, dtype: int64

Goldstein stats per dyad:
         count   mean    std   min    25%    50%    75%  max
dyad                                                        
CHN-AUS  279.0  1.880  3.045 -10.0  0.950  1.900  3.517  8.5
CHN-DEU  279.0  2.317  3.420 -10.0  1.000  2.500  4.158  8.0
CHN-FRA  294.0  2.449  2.846 -10.0  1.000  2.500  4.000  8.0
CHN-GBR  323.0  2.008  2.989 -10.0  1.000  2.400  4.000  8.0
CHN-IDN  268.0  2.341  3.146 -10.0  1.000  2.500  4.000  8.0
CHN-IND  307.0  1.791  2.893 -10.0  0.227  2.200  3.850  8.0
CHN-JPN  351.0  1.529  2.578 -10.0  0.000  1.825  3.012  8.0
CHN-PAK  325.0  2.668  2.642  -9.2  1.333  2.667  4.000  8.5
CHN-RUS  341

---
## Section 5: Phoenix NYT (CHN-USA only, 1990–2018)

NYT covers China-US relations densely. We include it only for CHN-USA.
For all other dyads it contributes 0 rows — this is correct by design.

In [72]:
NYT_PANEL_PATH = PHOENIX_DIR / "phoenix_nyt_chnusa_monthly_1990_2018.csv"

if NYT_PANEL_PATH.exists():
    print(f"Loading existing NYT panel: {NYT_PANEL_PATH}")
    df_nyt = pd.read_csv(NYT_PANEL_PATH)
    df_nyt['date'] = pd.to_datetime(df_nyt['date'])
    print(f"Shape: {df_nyt.shape}")
    print(f"Date range: {df_nyt['date'].min().date()} → {df_nyt['date'].max().date()}")
    print(f"\nDescriptive stats:")
    print(df_nyt.describe().round(3))
else:
    from tqdm import tqdm
    nyt_file = PHOENIX_DIR / "PhoenixBLN-NYT_1980-2018.csv"
    all_nyt  = []
    for chunk in tqdm(
        pd.read_csv(nyt_file, chunksize=200_000, low_memory=False,
                    usecols=['story_date','source','target','eid','goldstein']),
        desc="NYT CHN-USA"
    ):
        chunk['date']  = pd.to_datetime(chunk['story_date'], errors='coerce')
        chunk['year']  = chunk['date'].dt.year
        chunk['month'] = chunk['date'].dt.month
        chunk = chunk[(chunk['year']>=1990) & (chunk['year']<=2018)]
        if chunk.empty: continue
        mask = (
            (chunk['source'].str.contains('CHN',na=False) & chunk['target'].str.contains('USA',na=False)) |
            (chunk['source'].str.contains('USA',na=False) & chunk['target'].str.contains('CHN',na=False))
        )
        dc = chunk[mask].copy()
        if dc.empty: continue
        monthly = dc.groupby(['year','month']).agg(
            nyt_event_count    =('eid',       'count'),
            nyt_goldstein_mean =('goldstein', 'mean'),
            nyt_goldstein_std  =('goldstein', 'std'),
        ).reset_index()
        monthly['date'] = pd.to_datetime(monthly[['year','month']].assign(day=1))
        monthly['dyad'] = 'CHN-USA'
        all_nyt.append(monthly)

    df_nyt = pd.concat(all_nyt, ignore_index=True) if all_nyt else pd.DataFrame()
    if not df_nyt.empty:
        df_nyt = df_nyt.drop_duplicates(subset=['date','dyad'])
        df_nyt.to_csv(NYT_PANEL_PATH, index=False)
        print(f"Saved → {NYT_PANEL_PATH}  ({df_nyt.shape})")

Loading existing NYT panel: c:\Users\HP\Desktop\macro-geopolitics\data\03_nlp\phoenix\phoenix_nyt_chnusa_monthly_1990_2018.csv
Shape: (348, 5)
Date range: 1990-01-01 → 2018-12-01

Descriptive stats:
                                date  nyt_event_count  nyt_goldstein_mean  \
count                            348          348.000             348.000   
mean   2004-06-16 01:55:51.724137984           22.353               0.527   
min              1990-01-01 00:00:00            1.000              -3.800   
25%              1997-03-24 06:00:00           11.000              -0.342   
50%              2004-06-16 00:00:00           17.000               0.522   
75%              2011-09-08 12:00:00           27.000               1.398   
max              2018-12-01 00:00:00          142.000               4.800   
std                              NaN           20.101               1.413   

       nyt_goldstein_std  
count            346.000  
mean               4.302  
min                0.980  

In [73]:
# NYT Goldstein time series — quick visual
if 'df_nyt' in dir() and not df_nyt.empty:
    fig, axes = plt.subplots(2, 1, figsize=(14,6), sharex=True)
    axes[0].plot(df_nyt['date'], df_nyt['nyt_goldstein_mean'], color='darkorange', lw=0.9)
    axes[0].axhline(0, color='black', lw=0.5, ls='--')
    axes[0].set_ylabel('NYT Goldstein mean')
    axes[0].set_title('Phoenix NYT — CHN-USA monthly Goldstein score (1990–2018)', fontweight='bold')
    axes[1].bar(df_nyt['date'], df_nyt['nyt_event_count'], color='steelblue', width=25)
    axes[1].set_ylabel('Event count per month')
    axes[1].set_xlabel('Date')
    plt.tight_layout()
    plt.savefig(PHOENIX_DIR / 'explore_nyt_chnusa.png', dpi=130, bbox_inches='tight')
    plt.show()

C:\Users\HP\AppData\Local\Temp\ipykernel_10328\2428976284.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Section 6: Cross-Source Validation

Do GDELT, ICEWS, SWB, and NYT tell the same story?
We merge all four sources for CHN-USA and compute pairwise correlations
of their raw scores (before z-scoring). This is the best pre-composite check.

In [74]:
# Build a raw merged panel for CHN-USA
def load_if_exists(path):
    if Path(path).exists():
        df = pd.read_csv(path)
        df['date'] = pd.to_datetime(df['date'])
        return df
    return None

gdelt_src = df_gdelt[df_gdelt['dyad']=='CHN-USA'][['date','cameo_conflict_hits','cameo_cooperation_hits','total_volume']].copy()
gdelt_src['gdelt_net'] = (gdelt_src['cameo_conflict_hits'] - gdelt_src['cameo_cooperation_hits']) / gdelt_src['total_volume'].replace(0,np.nan)

icews_src = load_if_exists(ICEWS_DIR / "icews_monthly_panel_1995_2022.csv")
swb_src   = load_if_exists(PHOENIX_DIR / "phoenix_swb_monthly_1990_2019.csv")
nyt_src   = load_if_exists(PHOENIX_DIR / "phoenix_nyt_chnusa_monthly_1990_2018.csv")

merged = gdelt_src[['date','gdelt_net']].copy()
if icews_src is not None:
    icews_chnusa = icews_src[icews_src['dyad']=='CHN-USA'][['date','icews_goldstein_mean']]
    merged = merged.merge(icews_chnusa, on='date', how='left')
if swb_src is not None:
    swb_chnusa = swb_src[swb_src['dyad']=='CHN-USA'][['date','swb_goldstein_mean']]
    merged = merged.merge(swb_chnusa, on='date', how='left')
if nyt_src is not None:
    merged = merged.merge(nyt_src[['date','nyt_goldstein_mean']], on='date', how='left')

print(f"Merged CHN-USA panel shape: {merged.shape}")
print(f"\nMissingness per source:")
print(merged.isnull().mean().round(3))
print(f"\nPairwise Pearson correlations (raw scores):")
print(merged.drop(columns='date').corr().round(3))

Merged CHN-USA panel shape: (386, 5)

Missingness per source:
date                    0.000
gdelt_net               0.016
icews_goldstein_mean    0.187
swb_goldstein_mean      0.091
nyt_goldstein_mean      0.098
dtype: float64

Pairwise Pearson correlations (raw scores):
                      gdelt_net  icews_goldstein_mean  swb_goldstein_mean  \
gdelt_net                 1.000                -0.235              -0.010   
icews_goldstein_mean     -0.235                 1.000               0.222   
swb_goldstein_mean       -0.010                 0.222               1.000   
nyt_goldstein_mean       -0.076                 0.204               0.126   

                      nyt_goldstein_mean  
gdelt_net                         -0.076  
icews_goldstein_mean               0.204  
swb_goldstein_mean                 0.126  
nyt_goldstein_mean                 1.000  


In [75]:
# Time series overlay (normalised to zero mean for visual comparison)
fig, ax = plt.subplots(figsize=(15,5))
colors = {'gdelt_net':'#2c3e50','icews_goldstein_mean':'#e74c3c',
          'swb_goldstein_mean':'#27ae60','nyt_goldstein_mean':'#8e44ad'}
labels = {'gdelt_net':'GDELT','icews_goldstein_mean':'ICEWS',
          'swb_goldstein_mean':'SWB','nyt_goldstein_mean':'NYT'}

for col, color in colors.items():
    if col not in merged.columns: continue
    s = merged[col].dropna()
    s_norm = (s - s.mean()) / s.std()
    ax.plot(merged.loc[s.index,'date'], s_norm, lw=0.85, color=color,
            alpha=0.85, label=labels[col])

ax.axhline(0, color='grey', lw=0.5, ls='--')
ax.set_xlim(pd.Timestamp('1990-01-01'), pd.Timestamp('2022-03-01'))
ax.set_title('CHN-USA: all sources z-normalised overlay (positive = more conflict)', fontsize=12, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('z-score')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(GDELT_DIR / 'explore_source_overlay_chnusa.png', dpi=130, bbox_inches='tight')
plt.show()

C:\Users\HP\AppData\Local\Temp\ipykernel_10328\1777409495.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Section 7: FinBERT Sentiment Pipeline

### 7a. Model selection recap

Four models were benchmarked on 1,000 CHN-USA URL slugs. Winner:
`hakonmh/sentiment-xdistil-uncased` — highest non-neutral rate with correct
directional sensitivity (conflict text → negative, cooperation text → positive).

### 7b. Full corpus sentiment extraction

**Skipped per dyad if `sentiment_<DYAD>.csv` already exists.**
Files are written to `GDELT_DIR` (not `URLS_DIR`).

In [76]:
# Check which dyads already have sentiment CSVs
print("Sentiment file status:")
missing_dyads = []
for dyad in DYADS_ISO:
    out_path = GDELT_DIR / f"sentiment_{dyad}.csv"
    size = f"{out_path.stat().st_size/1e3:.0f} KB" if out_path.exists() else "MISSING"
    status = "OK " if out_path.exists() else "---"
    print(f"  {status}  {dyad:12s}  {size}")
    if not out_path.exists():
        missing_dyads.append(dyad)

print(f"\n{len(missing_dyads)} dyads still need processing: {missing_dyads if missing_dyads else 'none — all done!'}")

# Also check combined file
combined = GDELT_DIR / "sentiment_all_dyads_1990_2022.csv"
if combined.exists():
    print(f"\nCombined sentiment file: OK ({combined.stat().st_size/1e6:.1f} MB)")
else:
    print(f"\nCombined sentiment file: MISSING — run cell 7c to create it")

Sentiment file status:
  OK   CHN-USA       11 KB
  OK   CHN-JPN       11 KB
  OK   CHN-AUS       11 KB
  OK   CHN-FRA       11 KB
  OK   CHN-DEU       11 KB
  OK   CHN-GBR       11 KB
  OK   CHN-RUS       11 KB
  OK   CHN-IND       11 KB
  OK   CHN-IDN       11 KB
  OK   CHN-PAK       11 KB
  OK   CHN-VNM       11 KB

0 dyads still need processing: none — all done!

Combined sentiment file: OK (0.1 MB)


In [77]:
# Run sentiment extraction only for missing dyads
if missing_dyads:
    from transformers import pipeline
    from tqdm import tqdm
    import time

    MODEL_NAME = "hakonmh/sentiment-xdistil-uncased"
    print(f"Loading model: {MODEL_NAME}")
    sentiment_pipeline = pipeline("sentiment-analysis", model=MODEL_NAME, device=-1)
    print("Model loaded.\n")

    for dyad in missing_dyads:
        jsonl_file = URLS_DIR / f"corpus_{dyad}.jsonl"
        out_path   = GDELT_DIR / f"sentiment_{dyad}.csv"

        if not jsonl_file.exists():
            print(f"  SKIP {dyad} — no JSONL file found")
            continue

        print(f"Processing {dyad}...")
        month_scores       = defaultdict(list)
        month_non_neutral  = defaultdict(list)
        month_total        = defaultdict(int)

        total_lines = sum(1 for _ in open(jsonl_file, encoding='utf-8', errors='ignore'))
        with open(jsonl_file, encoding='utf-8', errors='ignore') as f:
            for line in tqdm(f, desc=f"  {dyad}", total=total_lines):
                try:
                    obj = json.loads(line)
                    date_str = obj.get('date','')
                    if not date_str:
                        dk = str(obj.get('day_key',''))
                        date_str = f"{dk[:4]}-{dk[4:6]}-01" if len(dk) >= 6 else ''
                    text = obj.get('cleaned_text','')
                    if not date_str or not text: continue

                    result  = sentiment_pipeline(text[:512])[0]
                    label   = result['label'].lower()
                    numeric = 1.0 if label=='positive' else (-1.0 if label=='negative' else 0.0)

                    month_scores[date_str].append(numeric)
                    month_total[date_str] += 1
                    if numeric != 0: month_non_neutral[date_str].append(numeric)
                except: continue

        rows = []
        for ds in sorted(month_scores):
            sc  = month_scores[ds]; nn = month_non_neutral[ds]; tot = month_total[ds]
            rows.append({'date': ds, 'dyad': dyad,
                         'sentiment_mean':          sum(sc)/tot,
                         'sentiment_intensity':      sum(abs(x) for x in sc)/tot,
                         'non_neutral_pct':          len(nn)/tot if tot else 0,
                         'non_neutral_sentiment_mean': sum(nn)/len(nn) if nn else 0,
                         'total_headlines':          tot})
        df_d = pd.DataFrame(rows)
        df_d['date'] = pd.to_datetime(df_d['date'])
        df_d.to_csv(out_path, index=False)
        print(f"  Saved {out_path.name}  ({len(df_d)} months)")
else:
    print("All sentiment files present — nothing to process.")

All sentiment files present — nothing to process.


In [78]:
# Combine all sentiment CSVs into single panel (skip if already done)
combined_path = GDELT_DIR / "sentiment_all_dyads_1990_2022.csv"

if combined_path.exists():
    print(f"Loading combined sentiment panel: {combined_path}")
    df_sent = pd.read_csv(combined_path)
    df_sent['date'] = pd.to_datetime(df_sent['date'])
else:
    all_sent = []
    for dyad in DYADS_ISO:
        fp = GDELT_DIR / f"sentiment_{dyad}.csv"
        if not fp.exists(): continue
        df = pd.read_csv(fp)
        df['date'] = pd.to_datetime(df['date'])
        all_sent.append(df)
    if all_sent:
        df_sent = pd.concat(all_sent, ignore_index=True)
        df_sent = df_sent.sort_values(['dyad','date'])
        df_sent.to_csv(combined_path, index=False)
        print(f"Saved combined panel → {combined_path}  ({df_sent.shape})")

# Summary
print(f"\nShape: {df_sent.shape}")
print(f"Dyads: {df_sent['dyad'].unique().tolist()}")
print(f"Date range: {df_sent['date'].min().date()} → {df_sent['date'].max().date()}")
print(f"\nMonths per dyad:")
print(df_sent.groupby('dyad')['date'].count())
print(f"\nSentiment stats (non_neutral_sentiment_mean by dyad):")
print(df_sent.groupby('dyad')['non_neutral_sentiment_mean'].describe().round(3))

Loading combined sentiment panel: c:\Users\HP\Desktop\macro-geopolitics\data\03_nlp\gdelt\sentiment_all_dyads_1990_2022.csv

Shape: (1177, 7)
Dyads: ['CHN-AUS', 'CHN-DEU', 'CHN-FRA', 'CHN-GBR', 'CHN-IDN', 'CHN-IND', 'CHN-JPN', 'CHN-PAK', 'CHN-RUS', 'CHN-USA', 'CHN-VNM']
Date range: 2013-04-01 → 2022-02-01

Months per dyad:
dyad
CHN-AUS    107
CHN-DEU    107
CHN-FRA    107
CHN-GBR    107
CHN-IDN    107
CHN-IND    107
CHN-JPN    107
CHN-PAK    107
CHN-RUS    107
CHN-USA    107
CHN-VNM    107
Name: date, dtype: int64

Sentiment stats (non_neutral_sentiment_mean by dyad):
         count   mean    std    min    25%    50%    75%    max
dyad                                                           
CHN-AUS  107.0 -0.291  0.262 -0.814 -0.472 -0.307 -0.125  0.464
CHN-DEU  107.0 -0.207  0.287 -0.850 -0.418 -0.209 -0.033  0.503
CHN-FRA  107.0 -0.244  0.278 -0.906 -0.448 -0.270 -0.040  0.373
CHN-GBR  107.0 -0.357  0.257 -0.801 -0.539 -0.397 -0.202  0.342
CHN-IDN  107.0 -0.021  0.351 -0.840 -0.22

In [79]:
# FinBERT sentiment time series — CHN-USA
sent_chnusa = df_sent[df_sent['dyad']=='CHN-USA'].sort_values('date') if 'df_sent' in dir() else None
if sent_chnusa is not None and not sent_chnusa.empty:
    fig, axes = plt.subplots(3,1, figsize=(14,9), sharex=True)
    axes[0].plot(sent_chnusa['date'], sent_chnusa['sentiment_mean'], lw=0.9, color='steelblue')
    axes[0].axhline(0, color='black', lw=0.5, ls='--')
    axes[0].set_title('FinBERT sentiment mean (all headlines, incl. neutral)', fontsize=10)
    axes[1].plot(sent_chnusa['date'], sent_chnusa['non_neutral_sentiment_mean'], lw=0.9, color='firebrick')
    axes[1].axhline(0, color='black', lw=0.5, ls='--')
    axes[1].set_title('FinBERT non-neutral sentiment mean (more variance — preferred for composite)', fontsize=10)
    axes[2].bar(sent_chnusa['date'], sent_chnusa['non_neutral_pct'], color='grey', width=25)
    axes[2].set_title('Proportion of non-neutral headlines per month', fontsize=10)
    axes[2].set_xlabel('Date')
    fig.suptitle('FinBERT Sentiment — CHN-USA (2013–2022)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(GDELT_DIR / 'explore_finbert_chnusa.png', dpi=130, bbox_inches='tight')
    plt.show()

C:\Users\HP\AppData\Local\Temp\ipykernel_10328\223540009.py:4: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axes = plt.subplots(3,1, figsize=(14,9), sharex=True)
C:\Users\HP\AppData\Local\Temp\ipykernel_10328\223540009.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Section 8: UCDP Dyadic Dataset (Evaluated, Excluded)

UCDP records state-based armed conflict (≥ 25 battle deaths/year).
China was not party to any such conflict with our 11 partner countries 1990–2022.
China's network centrality is effectively zero throughout, making UCDP
uninformative for this project. Confirmed below for completeness.

In [80]:
if UCDP_PATH.exists():
    df_ucdp = pd.read_csv(UCDP_PATH, low_memory=False)
    print(f"UCDP shape: {df_ucdp.shape}")
    print(f"Columns: {df_ucdp.columns.tolist()}")
    print(f"Years: {int(df_ucdp['year'].min())} – {int(df_ucdp['year'].max())}")

    # Check China presence (GWN code 710)
    def has_china(gwn):
        return pd.notna(gwn) and '710' in str(gwn).split(',')

    china_mask = df_ucdp['gwno_a'].apply(has_china) | df_ucdp['gwno_b'].apply(has_china)
    print(f"\nRows involving China (GWN=710): {china_mask.sum()}")
    if china_mask.sum() > 0:
        print(df_ucdp[china_mask][['year','gwno_a','gwno_b']].to_string())
    else:
        print("→ China absent from UCDP dyadic armed conflicts 1990–2022.")
        print("  This confirms exclusion from the composite index.")
else:
    print(f"UCDP file not found: {UCDP_PATH}")

UCDP shape: (3432, 25)
Columns: ['dyad_id', 'conflict_id', 'location', 'side_a', 'side_a_id', 'side_a_2nd', 'side_b', 'side_b_id', 'side_b_2nd', 'incompatibility', 'territory_name', 'year', 'intensity_level', 'type_of_conflict', 'start_date', 'start_prec', 'start_date2', 'start_prec2', 'gwno_a', 'gwno_a_2nd', 'gwno_b', 'gwno_b_2nd', 'gwno_loc', 'region', 'version']
Years: 1946 – 2024

Rows involving China (GWN=710): 28
      year gwno_a gwno_b
124   2008    710    NaN
583   1946    710    NaN
584   1947    710    NaN
585   1948    710    NaN
586   1949    710    NaN
714   1947    710    NaN
1079  1949    710    713
1080  1950    710    713
1081  1954    710    713
1082  1958    710    713
1262  1950    710    NaN
1263  1956    710    NaN
1264  1959    710    NaN
1647  1962    710    750
1648  1967    710    750
1649  2020    710    750
2108  1969    710    775
2109  1969    710    365
2672  1974    710    816
2673  1978    710    816
2674  1979    710    816
2675  1980    710    816
26

---
## Section 9: Pipeline Readiness Summary

Checks that all files expected by `04_instrument_diagnostics_and_macro_merge.ipynb` exist.

In [81]:
print("=" * 65)
print("PIPELINE READINESS CHECKLIST")
print("=" * 65)

checklist = {
    "GDELT lexical panel":      GDELT_DIR / "gdelt_birectional_lexical_panel.csv",
    "ICEWS monthly panel":      ICEWS_DIR / "icews_monthly_panel_1995_2022.csv",
    "Phoenix SWB panel":        PHOENIX_DIR / "phoenix_swb_monthly_1990_2019.csv",
    "Phoenix NYT CHN-USA":      PHOENIX_DIR / "phoenix_nyt_chnusa_monthly_1990_2018.csv",
    "FinBERT combined panel":   GDELT_DIR  / "sentiment_all_dyads_1990_2022.csv",
}

all_ok = True
for name, path in checklist.items():
    exists = path.exists()
    size   = f"{path.stat().st_size/1e3:.0f} KB" if exists else ""
    status = "✓" if exists else "✗ MISSING"
    print(f"  {status}  {name:<28} {size}")
    if not exists: all_ok = False

print()
if all_ok:
    print("All files present. 03c is ready to run.")
else:
    print("Missing files above must be generated before running 03c.")

# Quick row-count sanity
print("\n--- Row counts ---")
for name, path in checklist.items():
    if path.exists():
        try:
            df = pd.read_csv(path, nrows=0)
            n = sum(1 for _ in open(path)) - 1
            print(f"  {name:<28}: {n:>6,} rows")
        except: pass

PIPELINE READINESS CHECKLIST
  ✓  GDELT lexical panel          213 KB
  ✓  ICEWS monthly panel          227 KB
  ✓  Phoenix SWB panel            160 KB
  ✓  Phoenix NYT CHN-USA          20 KB
  ✓  FinBERT combined panel       118 KB

All files present. 03c is ready to run.

--- Row counts ---
  GDELT lexical panel         :  4,246 rows
  ICEWS monthly panel         :  3,488 rows
  Phoenix SWB panel           :  3,433 rows
  Phoenix NYT CHN-USA         :    348 rows
  FinBERT combined panel      :  1,177 rows


___
## Data Exploration & Validation – Summary

This notebook audits all raw sources used to construct the composite GPR index and confirms pipeline readiness.

### Data sources covered
- **GDELT** – lexical panel (1990–2022, 11 dyads, 4,246 rows) and URL corpus (3.2M headlines, 2013–2022)
- **ICEWS** – event files 1995–2022, aggregated to monthly dyadic panel (3,488 rows)
- **Phoenix SWB** – 1990–2019, 11 dyads (3,433 rows)
- **Phoenix NYT** – CHN‑USA only, 1990–2018 (348 rows)
- **UCDP** – evaluated and excluded (no China conflict with partner dyads, 1990–2022)

### Key validation outputs

1. **GDELT panel**  
   - Shape: (4246, 8) = 386 months × 11 dyads ✔  
   - Date range: 1990‑01‑01 → 2022‑02‑01  
   - Missing goldstein_mean: 137 rows (3.2%) – acceptable.

2. **ICEWS audit**  
   - 27 yearly files, columns verified (`Source Country`, `Target Country`, `Intensity`).  
   - Russia appears as `"Russian Federation"` (partial match works).  
   - Full dyad×year audit saved → heatmap shows no zero cells after 1995.  
   - Monthly panel shape: (3488, 7) – dyads have 304–324 months each.

3. **Phoenix SWB**  
   - Actor codes contain ISO3 (e.g., `CHNGOV`, `RUS`). `str.contains()` works.  
   - Dyad×year audit saved – CHN‑USA has 1990–2019 coverage; some dyads (e.g., CHN‑IDN) are sparse.  
   - Monthly panel shape: (3433, 7).  

4. **Phoenix NYT**  
   - Only CHN‑USA. Shape: (348, 5), 1990‑2018. Goldstein mean ranges from −3.8 to +4.8.

5. **Cross‑source correlation (CHN‑USA raw scores)**  
   - GDELT net conflict vs ICEWS Goldstein: −0.235  
   - SWB vs ICEWS: +0.222  
   - NYT vs ICEWS: +0.204  
   - Low correlations reflect different definitions and coverage – justifies composite approach.

6. **FinBERT sentiment**  
   - Model: `hakonmh/sentiment-xdistil-uncased` (highest non‑neutral rate).  
   - Processed all 11 dyads (2013‑2022, 107 months each).  
   - Output: `sentiment_all_dyads_1990_2022.csv` (1,177 rows, 7 columns).  
   - CHN‑USA non‑neutral sentiment mean: −0.403 (negative bias, plausible).

7. **UCDP**  
   - No dyadic armed conflict involving China with any of the 11 partners (1990‑2022).  
   - Confirmed exclusion from composite index.

### Pipeline readiness
All files required by `04_instrument_diagnostics_and_macro_merge.ipynb` are present:
- GDELT lexical panel (213 KB)  
- ICEWS monthly panel (227 KB)  
- SWB monthly panel (160 KB)  
- NYT CHN‑USA panel (20 KB)  
- FinBERT combined panel (118 KB)  

**Row counts:** 4,246 / 3,488 / 3,433 / 348 / 1,177 – ready for merging.

### Note on data storage
The raw corpora (GDELT JSONL, ICEWS yearly zip files, Phoenix CSVs) total >4 GB. They are **not uploaded to GitHub** (excluded via `.gitignore`). The notebook caches all derived panels (`*_audit_counts.csv`, `*_monthly_panel_*.csv`, `sentiment_*.csv`), which are small enough to version or share separately. To re‑run from scratch, place the original files in the expected directories (`data/03_nlp/gdelt/`, `icews/`, `phoenix/`).

